# Step 2 — The LSTM language model

Step 1 left two token arrays and a 20,000-word vocabulary. This notebook builds the model that
predicts the next token, and trains it.

Runs in two places: locally on CPU with `SUBSET_TOKENS` set, to prove the pipeline works, and on a
Colab GPU with `SUBSET_TOKENS = None` for the real run. The Colab section at the bottom explains why
that split exists and what has to be true for the GPU path to actually be fast.

In [ ]:
import os, json, time, numpy as np, tensorflow as tf

IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/sequential_model"
    OUT  = DATA
else:
    DATA = "../data/processed"
    OUT  = "../data/outputs"
os.makedirs(OUT, exist_ok=True)

SEQ_LEN       = 100
BATCH_SIZE    = 128 if not IN_COLAB else 256
EMBED_DIM     = 256
LSTM_UNITS    = 512
SUBSET_TOKENS = 4_000_000 if not IN_COLAB else None    # None = whole corpus

meta  = json.load(open(f"{DATA}/vocab.json"))
itos  = meta["itos"]; VOCAB = len(itos)
train = np.load(f"{DATA}/train_tokens.npy")
val   = np.load(f"{DATA}/val_tokens.npy")
if SUBSET_TOKENS:
    train, val = train[:SUBSET_TOKENS], val[:SUBSET_TOKENS // 20]

print(f"{'COLAB' if IN_COLAB else 'LOCAL'} | GPU: {tf.config.list_physical_devices('GPU') or 'none'}")
print(f"vocab {VOCAB:,} | train {train.size:,} tokens | val {val.size:,} tokens")

## Building the input pipeline without materialising it

The naive version builds `X` and `Y` as arrays of shape `(n_windows, SEQ_LEN)`. At 3.2 million
windows that is **1.3 GB each**, before the model exists — and on the full corpus it is the first
thing that would fail.

Instead the flat token array is reshaped into non-overlapping windows of `SEQ_LEN + 1`, which for a
contiguous array is a **view, not a copy**, and the input/target split happens inside the
`tf.data` pipeline: input is the window minus its last token, target is the window minus its first.
Predicting the next token at every position is what makes this efficient — one pass produces
`SEQ_LEN` training signals, not one.

In [ ]:
def make_dataset(tokens, batch_size, shuffle):
    n = (tokens.size - 1) // (SEQ_LEN + 1)
    windows = tokens[: n * (SEQ_LEN + 1)].reshape(n, SEQ_LEN + 1)   # view, no copy
    ds = tf.data.Dataset.from_tensor_slices(windows)
    if shuffle:
        ds = ds.shuffle(10_000, seed=42, reshuffle_each_iteration=True)
    ds = ds.map(lambda w: (tf.cast(w[:-1], tf.int32), tf.cast(w[1:], tf.int32)),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train, BATCH_SIZE, shuffle=True)
val_ds   = make_dataset(val,   BATCH_SIZE, shuffle=False)

for x, y in train_ds.take(1):
    print("input ", x.shape, x.dtype)
    print("target", y.shape)
    print("shifted by one:", " ".join(itos[i] for i in x[0, :10].numpy()))
    print("                ", " ".join(itos[i] for i in y[0, :10].numpy()))
    break

## The model

```
Embedding(20000, 256) → LSTM(512, return_sequences=True) → Dense(20000)
```

**`Embedding`** maps each token id to a 256-dimensional learned vector. Unlike the TF-IDF project,
nothing here is hand-designed: the model discovers what dimensions are worth having.

**`LSTM`** is the point of the project. A plain RNN multiplies its hidden state by the same matrix at
every step, so gradients flowing back through many steps either vanish or explode — it cannot connect
a pronoun to a noun forty tokens earlier. An LSTM adds a **cell state** that runs through the
sequence with only *additive* interactions, plus three gates that learn what to erase (forget), what
to write (input), and what to expose (output). The cell state is the "conveyor belt" in
[Olah's article](https://colah.github.io/posts/2015-08-Understanding-LSTMs/); the additive path is
why the gradient survives.

**`return_sequences=True`** emits a prediction at *every* timestep rather than only the last, which
is what makes a 100-token window produce 100 training examples.

**`Dense(VOCAB)` with no activation**, paired with `from_logits=True` in the loss. Computing softmax
inside the loss is numerically stabler than producing probabilities and taking their log.

**No `mask_zero`, no `recurrent_dropout`** — both deliberate, both about the GPU path. See below.

In [ ]:
def build_model(batch_size=None):
    return tf.keras.Sequential([
        tf.keras.layers.Input(batch_shape=(batch_size, SEQ_LEN) if batch_size
                              else (None, SEQ_LEN)),
        tf.keras.layers.Embedding(VOCAB, EMBED_DIM),       # mask_zero=False on purpose
        tf.keras.layers.LSTM(LSTM_UNITS, return_sequences=True,
                             recurrent_dropout=0.0),        # 0.0 required for cuDNN
        tf.keras.layers.Dense(VOCAB),
    ])

model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)
model.summary()

### Why `mask_zero` is off, having been the thing to watch

Padding is a real hazard for sequence models: pad with zeros, say nothing about it, and the model is
trained to predict padding after real text — which a *generative* model will then happily generate.
`Embedding(..., mask_zero=True)` fixes that, and it genuinely works: training a small model and then
changing the labels **only at padded positions** moves the loss from 3.3587 to 3.8899 without
masking, and not at all (3.8941 → 3.8941) with it. The mask reaches the loss.

It is off here because **this pipeline produces no padding**. Fixed-length windows over a
concatenated stream are all exactly `SEQ_LEN` long; the mask would be all-`True` at every position,
costing time to compute and hiding nothing.

On GPU it would be worse than free. The cuDNN LSTM kernel — the reason a GPU is 10–30× faster here
rather than 2× — has conditions, and two are relevant:

> 3. `recurrent_dropout` == 0
> 6. Inputs, if use masking, are strictly right-padded.

Miss either and Keras silently falls back to the generic implementation, with no warning and an
order of magnitude less speed. `mask_zero=False` and `recurrent_dropout=0.0` are there to keep that
path open. Index 0 is still reserved in the vocabulary, so masking can be switched on if the
sequence construction ever changes to per-sentence batches.

## Training

Checkpoint every epoch, and log to CSV, because a Colab session can disconnect and an interrupted run
should cost one epoch rather than everything.

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(f"{OUT}/lstm_lm.keras", save_best_only=True,
                                       monitor="val_loss"),
    tf.keras.callbacks.CSVLogger(f"{OUT}/training_log.csv", append=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                     restore_best_weights=True),
]

EPOCHS = 3 if SUBSET_TOKENS else 1
t0 = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    callbacks=callbacks, verbose=2)
train_secs = time.time() - t0
print(f"\n{train_secs/60:.1f} min for {EPOCHS} epoch(s) on {train.size:,} tokens")
print(f"throughput: {EPOCHS*train.size/train_secs:,.0f} tokens/s")

## Perplexity

Cross-entropy in nats is hard to feel. **Perplexity** = `exp(loss)` is the same number expressed as
"how many equally-likely tokens is the model effectively choosing between". A perplexity of 20,000
means it has learned nothing at all — that is the vocabulary size, i.e. uniform guessing. Anything
below that is progress, and the scale is multiplicative: 200 is not "10% better" than 220.

In [ ]:
val_loss = model.evaluate(val_ds, verbose=0)
print(f"val loss       {val_loss:.4f} nats")
print(f"val perplexity {np.exp(val_loss):,.1f}")
print(f"uniform guess  {VOCAB:,.1f}   (what 'learned nothing' looks like)")

json.dump({"val_loss": float(val_loss), "val_perplexity": float(np.exp(val_loss)),
           "train_secs": round(train_secs, 1), "epochs": EPOCHS,
           "train_tokens": int(train.size), "vocab": VOCAB,
           "subset": SUBSET_TOKENS, "colab": IN_COLAB},
          open(f"{OUT}/lm_metrics.json", "w"), indent=1)

## Running the real thing on Colab

The local configuration above deliberately trains on a 4M-token slice. The full corpus is 325M
tokens, and on this CPU that is roughly **9–10 hours per epoch** — a benchmark at vocabulary 50,000
did not finish inside ten minutes, which is what forced the vocabulary decision in Step 1.

This is the case where a GPU is genuinely the right answer, and it is worth being precise about why,
because the previous project (`260106_DeepLearningForNlp`) reached the opposite conclusion. There the
model was a two-layer MLP on TF-IDF features: 4 seconds per epoch on CPU, GPU irrelevant. Here it is
a recurrent model over hundreds of millions of timesteps, and cuDNN fuses the whole recurrence into
one kernel. The distinction is not "neural network" versus "not" — it is whether the computation is
a few big matrix multiplications or millions of small sequential ones.

**To run it:**

1. Upload `train_tokens.npy` (651 MB), `val_tokens.npy` (17 MB) and `vocab.json` to a Drive folder
   named `sequential_model`. Only these three — not the 1.73 GB of source books, which Step 1 has
   already consumed.
2. Open this notebook in Colab, Runtime → Change runtime type → **T4 GPU**.
3. Run all. `IN_COLAB` switches the paths, raises the batch size to 256, and sets
   `SUBSET_TOKENS = None`.

**Confirm the cuDNN path is actually being used** before walking away — this is the single check
worth doing, because falling back is silent:

```python
# a fast epoch is ~1000x this; if the numbers are close, cuDNN is NOT engaged
%timeit -n1 -r1 model.fit(train_ds.take(50), epochs=1, verbose=0)
```

Free-tier Colab disconnects after roughly 12 hours, and sooner if the tab is idle. Checkpoints go to
Drive after every epoch, so a disconnect costs the current epoch rather than the run.